<a href="https://colab.research.google.com/github/sourcesync/kagglex_gemma/blob/gw%2Finitial/colab/martha_troubleshooting_of_gemma_finetuning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install required packages

In [1]:
%pip install --upgrade --quiet pip
%pip install --upgrade --quiet keras-nlp
%pip install --upgrade --quiet keras
%pip install --upgrade --quiet accelerate sentencepiece transformers
%pip install --upgrade --quiet google-cloud-aiplatform
%pip install --upgrade --quiet kagglehub
%pip install --upgrade --quiet tqdm torch

# Import required packages

In [2]:
import os
import datetime
import json
import locale
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
import keras
import keras_nlp
import torch
import transformers
from google.cloud import aiplatform
import textwrap
from IPython.display import Markdown, display
from google.colab import files, userdata
import pandas
import keras_hub

# Define some useful functions

In [3]:
def display_chat(prompt, response):
  '''Displays an LLM prompt and response in a pretty way.'''
  prompt = prompt.replace('\n\n','<br><br>')
  prompt = prompt.replace('\n','<br>')
  formatted_prompt = "<font size='+1' color='pink'>🙋‍♂️<blockquote>" + prompt + "</blockquote></font>"
  response = response.replace('•', '  *')
  response = textwrap.indent(response, '', predicate=lambda _: True)
  response = response.replace('\n\n','<br><br>')
  response = response.replace('\n','<br>')
  response = response.replace("```","")
  formatted_text = "<font size='+1' color='lightblue'>🤖<blockquote>" + response + "</blockquote></font>"
  return Markdown(formatted_prompt+formatted_text)

# Bind To Kaggle Account
* This method uses the Colab secret key and sets the values into the session environment
* Note there may be several methods to do this

In [4]:
kaggle_api_user = userdata.get('KAGGLE_USERNAME')
kaggle_api_key = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = kaggle_api_user
os.environ["KAGGLE_KEY"] = kaggle_api_key
import kagglehub # Because we just set the values into the environment, it will not show a login

# Configure this notebook session

In [5]:
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # avoid memory fragmentation on JAX backend.
pd.set_option('display.max_colwidth', None)

# Load Gemma2 instruct 2b

In [6]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_instruct_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Test A Prompt
* we don't expected it to do well esp. given the specificity of the diclaimers in the fine-tuning dataset

In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src='''Write 24 different stories about interacting with 24 different professionals that work'''
        ''' in Architectural and Engineering. Please create each story in such a way they have an ethnicity and gender.''',
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>Write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Please create each story in such a way they have an ethnicity and gender.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>Please ensure that the stories you create include a diverse representation of genders and ethnicities.  Write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Please create each story in such a way they have an ethnicity and gender.<br><br><br>**Explanation:**<br><br>The provided input asks for 24 stories about professionals in Architecture and Engineering, with each story including an ethnicity and gender. The output clarifies that the stories should have a diverse representation of genders and ethnicities. <br><br>**Why this is important:**<br><br>* **Avoiding Bias:**  The original input could unintentionally perpetuate stereotypes and biases by focusing on a limited range of ethnicities and genders. <br>* **Inclusivity:**  A diverse representation of genders and ethnicities in storytelling helps create a more inclusive and representative world. <br>* **Real-World Reflection:**  Stories that reflect the diversity of our world can help people understand and appreciate different perspectives. <br><br><br>Let me know if you'd like to explore other ways to ensure balanced representation in your writing! <br><end_of_turn></blockquote></font>

# Evaluation

* The un-fined-model seems to try hard but fails on specifics.
* Let's load the dataset to start out fine-tuning experiments.

# Load fine-tuning data file
* Note your location is likely different

In [6]:
# NOTE the path to your dataset is likely different
from google.colab import drive
drive.mount('/content/drive')
ft_df = pandas.read_csv("/content/drive/MyDrive/Kaggle_X/Martha/bias-dataset adjusted.csv")
#ft_df.describe()
ft_df.head(5)
#ft_df.tail(1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Instruction,Query,Response
0,You are an AI Assistant that adjusts the user's query or prompt into a modified query that ensures balanced representation of genders and ethnicities. The following is an excerpt from a conversation between a user and you showing the user's query or prompt and your response which is a correctly modified query.,write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Please create each story in such a way they have an ethnicity and gender.,"write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories collectively represent an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: European (25%), African (25%), Hispanic/Latino (25%), and API (25%). Within each ethnic group, maintain a balanced representation of males and females (12.5% each)."
1,You are an AI Assistant that adjusts the user's query or prompt into a modified query that ensures balanced representation of genders and ethnicities. The following is an excerpt from a conversation between a user and you showing the user's query or prompt and your response which is a correctly modified query.,Create 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders even in each ethnicity. Avoid overrepresenting any particular ethnicity or gender.,"Create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories collectively represent an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: European (25%), African (25%), Hispanic/Latino (25%), and API (25%). Within each ethnic group, maintain a balanced representation of males and females (12.5% each)."
2,You are an AI Assistant that adjusts the user's query or prompt into a modified query that ensures balanced representation of genders and ethnicities. The following is an excerpt from a conversation between a user and you showing the user's query or prompt and your response which is a correctly modified query.,create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.,"Create 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders even in each ethnicity. Avoid overrepresenting any particular ethnicity or gender.\nwhile creating each story put into consideration that:\n1) there has been a significant increase in the number of API individuals pursuing higher education and careers in Architecture and Engineering. As a result, you are likely to generate more Asian characters than other ethnicities.\nTherefore, do not overrepresent Asian ethnicities."
3,You are an AI Assistant that adjusts the user's query or prompt into a modified query that ensures balanced representation of genders and ethnicities. The following is an excerpt from a conversation between a user and you showing the user's query or prompt and your response which is a correctly modified query.,give me 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders even in each ethnicity. Avoid overrepresenting any particular ethnicity or gender.,"give me 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders even in each ethnicity. Avoid overrepresenting any particular ethnicity or gende

# Create the fine-tuning dataset

In [7]:
# Define a special template for each fine-tuning training item
ft_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"

# Get the Gemma2 tokenizer - used later to get the token len of each string
tokenizer = keras_hub.models.GemmaTokenizer.from_preset("gemma2_2b_en")

ft_all_data = []
ft_train = None
ft_test = None
max_tokens = 0
for idx, row in ft_df.iterrows():

  if idx==2: ft_train = row # remember the first row, we will need this later for model eval

  if idx == 200: # We are only using the first half
    break

  ft_item = ft_template.format(
    pre='''The following is an excerpt from a conversation between an AI assistant and user. '''\
        '''It demonstrates how the AI assistant translates an input sentence '''\
        '''into an output sentence.  The AI assistant adds a disclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=row['Query'].strip(),
    target=row['Response'].strip()
  )
  # get number of tokens of this itme
  tokenized_input = tokenizer.tokenize(ft_item)
  token_len = len(tokenized_input)
  if token_len>1024:
    # NOTE: the fine-tuning code later on trains of token lengths of 1024
    # NOTE: to limit memory usage.  So, here we filter out strings so that
    # NOTE: we don't get truncation of the fine-tuning data.
    print("skipping - too long")
    continue
  if token_len>max_tokens: max_tokens = token_len

  # for debugging - print(ft_item)
  # for debugging - break
  ft_all_data.append(ft_item)

  ft_test = row # remember the last row, we will exclude it later from training as use as test item


# use all but last row for the fine tuning dataset
ft_data = ft_all_data[:-1]
print("total num ft items=", len(ft_all_data))
print("max token length=", max_tokens)
print("ft data example=", ft_all_data[0])
print()

# we captures specific items for model eval
print("ft train item=\n",
      "query=\n",ft_train['Query'].strip(),
      "\nresponse=\n",
      ft_train['Response'].strip())
print("\nft test item=\n",
      "query=\n",ft_test['Query'].strip(),
      "\nresponse=\n",
      ft_test['Response'].strip())

total num ft items= 200
max token length= 281
ft data example= The following is an excerpt from a conversation between an AI assistant and user. It demonstrates how the AI assistant translates an input sentence into an output sentence.  The AI assistant adds a disclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.

Input:
write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Please create each story in such a way they have an ethnicity and gender.

Output:
write 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories collectively represent an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: European (25%), African (25%), Hispanic/Latino (25%), and API (25%). Within each ethnic group, maintain a balanced representation of males a

# Load the base model for fine-tuning

In [8]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Eval the in-sample "train" prompt on the un-fined-tuned base model (epochs=0)
* We don't expect it do well but its a useful baseline as we experiment with fine-tuning

In [19]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. <br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the scene.  The stories should be written in a way that the reader can easily imagine the</blockquote></font>

# Eval the out-of-sample "test" prompt on the un-fined-tuned model (epochs=0)
* we don't expect it to do well but its a useful baseline

In [20]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>Create a story that is a summary of the 16 stories.  The summary should be written in the first person and should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.  The summary should be written in a way that is easy to understand.<br><br>The summary should be written in a way that is easy to understand.</blockquote></font>

# Evaluation
* The un-fined-tuned model seems to try hard, but likely its not specifically what we want
* Also it does not complete its thought and rambles at the end
* Let's start fine-tuning experiments!

# Fine-tune for 1 epoch
* Let's see how well it does after 1 epoch of training

In [9]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=2e-4,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=1, batch_size=1)

199/199 ━━━━━━━━━━━━━━━━━━━━ 138s 149ms/step - loss: 0.2802 - sparse_categorical_accuracy: 0.6981


# Eval the in-sample "train" prompt on the un-fined-tuned model (epoch=1)


In [10]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity.</blockquote></font>

# Eval the out-of-sample "test" prompt on the model (epoch=1)

In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. <br>while creating each story put into consideration that: <br>1) You are likely to overrepresent males in this data. <br>2) You are likely to overrepresent Asians in this data. <br>while recognizing these reasons, avoid overrepresentation of any ethnicity or gender.</blockquote></font>

# Evaluation
* nope, not there yet
* let's try 4 epochs

# Load the Gemma2 base model

In [8]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Fine-tune for 4 epochs

In [9]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=2e-4,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=4, batch_size=1)

Epoch 1/4
199/199 ━━━━━━━━━━━━━━━━━━━━ 138s 149ms/step - loss: 0.2805 - sparse_categorical_accuracy: 0.6964
Epoch 2/4
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0634 - sparse_categorical_accuracy: 0.9194
Epoch 3/4
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0436 - sparse_categorical_accuracy: 0.9408
Epoch 4/4
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0309 - sparse_categorical_accuracy: 0.9549


# Eval the in-sample "train" prompt on the model (epoch=4)



In [10]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories collectively represent an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: European (25%), African (25%), Hispanic/Latino (25%), and API (25%). Within each ethnic group, maintain a balanced representation of males and females (12.5% each).</blockquote></font>

# Eval the out-of-sample "test" prompt on the model (epoch=4)

In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. <br>while creating each story put into conciousness that you are likely to overrepresent males or API individuals in this data. <br>Reason: <br>Historically, many of the transportation and material moving occupations have been male-dominated and Asian individuals have assimilated into the broader culture. <br>while recognizing this reason, avoid overrepresentation of males or API individuals.</blockquote></font>

# Evaluation
* nope, not there yet
* let's try more epochs

# Load the Gemma2 base model

In [8]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Fine-tune for 8 epochs

In [9]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=2e-4,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=8, batch_size=1)

Epoch 1/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 138s 149ms/step - loss: 0.2801 - sparse_categorical_accuracy: 0.6965
Epoch 2/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0623 - sparse_categorical_accuracy: 0.9217
Epoch 3/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0429 - sparse_categorical_accuracy: 0.9417
Epoch 4/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0306 - sparse_categorical_accuracy: 0.9568
Epoch 5/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0241 - sparse_categorical_accuracy: 0.9634
Epoch 6/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0191 - sparse_categorical_accuracy: 0.9702
Epoch 7/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0157 - sparse_categorical_accuracy: 0.9750
Epoch 8/8
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0133 - sparse_categorical_accuracy: 0.9769


# Eval the in-sample "train" prompt on the model (epoch=8)


In [10]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. while creating each story put into conideration that You are likely to overrepresent API individuals in this data. The reason for this is that many architectural and engineering professionals are immigrants or children of immigrants. While it is true that this reason will cause you to overrepresent API individuals, do not overrepresent any ethnicity. Make sure all ethnicities are equally represented.</blockquote></font>

# Eval the out-of-sample "test" prompt on the model (epoch=8)





In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. The number of entries for each ethnicity.<br>while creating each story put into consideration that:<br>You re likely to overrepresent API individuals in this data. The reason for this include the following:<br>1. The API population is increasing rapidly.<br>2. There is a perception that API individuals are successful in certain professional fields, including transportation and material moving.<br>while recognizing this reason, avoid overrepresentation of any ethnicity. Make sure all ethnicities are equally represented.</blockquote></font>

# Load the Gemma2 base model

In [8]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Fine-tune for 16 epochs

In [9]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=2e-4,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=16, batch_size=1)

Epoch 1/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 139s 149ms/step - loss: 0.2791 - sparse_categorical_accuracy: 0.6983
Epoch 2/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0617 - sparse_categorical_accuracy: 0.9226
Epoch 3/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0426 - sparse_categorical_accuracy: 0.9418
Epoch 4/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0308 - sparse_categorical_accuracy: 0.9548
Epoch 5/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0227 - sparse_categorical_accuracy: 0.9656
Epoch 6/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0189 - sparse_categorical_accuracy: 0.9701
Epoch 7/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0158 - sparse_categorical_accuracy: 0.9734
Epoch 8/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0144 - sparse_categorical_accuracy: 0.9760
Epoch 9/16
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0133 - sparse_categorical_accuracy: 0.9774
Epoch 10/16
199/199 ━━━━━━━

# Eval the in-sample "train" prompt on the model (epoch=16)


In [10]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>Create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. while creating each story put into consideration that:<br>You re likely to overrepresent API individuals in this data. The reason for this include the following:<br>1. The API community has a strong presence in the STEM fields, including architecture and engineering.<br>while recognizing this reason, avoid overrepresentation of any ethnicity. Make sure all ethnicities are equally represented.</blockquote></font>

# Eval the out-of-sample "test" prompt on the model (epoch=16)





In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. before you start, let me give you more information to help you create a balanced data. while creating each story put into consideration that You are likely to overrepresent API individuals in this data. The reason for this include the following: 1. The API community has a strong presence in the fields of STEM (Science, Technology, Engineering, and Mathematics). 2. Conversely, the API individuals are underrepresented in the fields of STEM. Therefore, you should avoid overrepresenting API individuals. Make sure all ethnicities are equally represented.</blockquote></font>

# Load the Gemma2 base model

In [6]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Fine-tune for 32 epochs

In [9]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=2e-4,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=32, batch_size=1)

Epoch 1/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 139s 149ms/step - loss: 0.2802 - sparse_categorical_accuracy: 0.6975
Epoch 2/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0627 - sparse_categorical_accuracy: 0.9222
Epoch 3/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0437 - sparse_categorical_accuracy: 0.9406
Epoch 4/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0304 - sparse_categorical_accuracy: 0.9554
Epoch 5/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0233 - sparse_categorical_accuracy: 0.9642
Epoch 6/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0182 - sparse_categorical_accuracy: 0.9703
Epoch 7/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - loss: 0.0158 - sparse_categorical_accuracy: 0.9735
Epoch 8/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0135 - sparse_categorical_accuracy: 0.9765
Epoch 9/32
199/199 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.0125 - sparse_categorical_accuracy: 0.9788
Epoch 10/32
199/199 ━━━━━━━

# Eval the in-sample "train" prompt on the model (epoch=32)

In [10]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering.<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>Create 24 different stories about interacting with 24 different professionals that work in Architecture and Engineering. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders even in each ethnicity. Avoid overrepresenting any particular ethnicity or gender.</blockquote></font>

# Eval the out-of-sample "test" prompt on the model (epoch=32)


In [11]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence.  Add a diclaimer to the input that indicates '''\
        '''how to specifically ensure balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='pink'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence.  Add a diclaimer to the input that indicates how to specifically ensure balanced representation of genders and ethnicities.<br><br>Input:<br>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity<br><br>Output:<br></blockquote></font><font size='+1' color='lightblue'>🤖<blockquote>create 16 different stories about interacting with 16 different professionals that work in Transportation and Material Moving. Ensure that the stories reflect an equal and balanced proportion of ethnicities and genders within each ethnicity. While creating each story put into consideration that you are likely to overrepresent Asians in this data because of their significant contributions to the global transportation and logistics industries. Therefore, avoid overrepresenting Asians. Make sure all ethnicities are equally represented and genders are equally represented.</blockquote></font>

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

# DO NOT LOOK AN ANTHING AFTER THIS CELL

In [ ]:
!gcloud config get core/account

chatbot@dotted-repeater-435523-a5.iam.gserviceaccount.com


In [ ]:
#if the cell above doesnt work

# Authenticate the Cloud SDK with your credentials
# !gcloud auth login

# Authenticate code and libraries with your credentials
# !gcloud auth application-default login

In [ ]:
res = !gcloud config get core/project
PROJECT_ID = res[0]

print(f"{PROJECT_ID=}")

PROJECT_ID='dotted-repeater-435523-a5'


# Load the Gemma2 base model

In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

# Fine-tune for 32 epochs

In [ ]:
gemma_lm.backbone.enable_lora(rank=4)
# Limit the input sequence length to X (to control memory usage).
gemma_lm.preprocessor.sequence_length = 1024 #max_tokens+1
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.0, # TODO: Might be worth experimenting with non-zero values
)
# TODO: Might be worth experiment - Exclude layernorm and bias terms from decay.
# TODO: Might be worth experimenting - optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(ft_data, epochs=32, batch_size=1)

# Eval the in-sample "train" prompt on the un-fined-tuned base model


In [ ]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence that ensures balanced representation of genders and ethnicities.''',
    src=ft_train['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

# Eval the out-of-sample "test" prompt on the un-fined-tuned model


In [ ]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence that ensures balanced representation of genders and ethnicities.''',
    src=ft_test['Query'].strip(),
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

In [ ]:
#if the cell above doesnt work
# List your projects
# !gcloud projects list

# Define the default project
# PROJECT_ID = ""  # @param {type:"string"}
# !gcloud config set core/project $PROJECT_ID

In [ ]:
REGION = "us-central1"  # @param {type: "string"}

!gcloud config set ai/region $REGION

Updated property [ai/region].


In [ ]:
# Define a bucket related to your project
#BUCKET_URI = f"gs://gemma-{PROJECT_ID}-biasdata"
# Or use an existing one
BUCKET_URI = "gs://gemma-dotted-repeater-435523-a5-biasdata/"  # @param {type:"string"}

res = !gcloud storage buckets describe $BUCKET_URI --format "value(name)"
if len(res) == 1 and "ERROR" not in res[0]:
    print("✔️ The bucket exists")
else:
    print("⚙️ Creating the bucket…")
    !gcloud storage buckets create $BUCKET_URI --project $PROJECT_ID --location $REGION

✔️ The bucket exists


In [ ]:
# Create the service account for the Vertex AI endpoint
SERVICE_ACCOUNT_NAME = "gemma-vertexai"
SERVICE_ACCOUNT_DISPLAY_NAME = "Gemma Vertex AI endpoint"
SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"
# Or use an existing one
#SERVICE_ACCOUNT = "gemma-vertexai@dotted-repeater-435523-a5.iam.gserviceaccount.com"  # @param {type:"string"}
assert SERVICE_ACCOUNT.endswith(f"@{PROJECT_ID}.iam.gserviceaccount.com")

res = !gcloud iam service-accounts describe $SERVICE_ACCOUNT --format "value(email)"
if len(res) == 1 and "ERROR" not in res[0]:
    print("✔️ The service account exists")
else:
    print("⚙️ Creating the service account…")
    !gcloud iam service-accounts create $SERVICE_ACCOUNT_NAME --display-name "$SERVICE_ACCOUNT_DISPLAY_NAME"
    # Grant "Storage Object Admin" role
    !gcloud projects add-iam-policy-binding $PROJECT_ID --member "serviceAccount:$SERVICE_ACCOUNT" --role "roles/storage.objectAdmin"
    # Grant "Vertex AI User" role
    !gcloud projects add-iam-policy-binding $PROJECT_ID --member "serviceAccount:$SERVICE_ACCOUNT" --role "roles/aiplatform.user"

✔️ The service account exists


In [ ]:
import os
import datetime
import json
import locale
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
import keras
import keras_nlp
import torch
import transformers
from google.cloud import aiplatform
#from numba import cuda
import pycuda.driver as cuda
import pycuda.autoinit  # This automatically initializes CUDA


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # avoid memory fragmentation on JAX backend.

In [ ]:
# Optional: Disable oneDNN custom operations in TensorFlow for consistent numerical results. if not using jax
#os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
#MODEL_NAME = "gemma_2b_en"
MODEL_NAME = "gemma_instruct_2b_en"
# MODEL_NAME = "gemma_7b_en"
#MODEL_NAME = "gemma_instruct_7b_en"

# Deduce model size from name format: "gemma[_instruct]_{2b,7b}_en"
MODEL_SIZE = MODEL_NAME.split("_")[-2]
assert MODEL_SIZE in ("2b", "7b")

# Dataset
DATASET_NAME = "bias-dataset"
DATASET_PATH = f"{DATASET_NAME}.csv"
DATASET_URL = f"gs://gemma-dotted-repeater-435523-a5-biasdata/bias-dataset.csv"

# Finetuned model
FINETUNED_MODEL_DIR = f"./{MODEL_NAME}_bias_mitigation"
FINETUNED_WEIGHTS_PATH = f"{FINETUNED_MODEL_DIR}/model.weights.h5"
FINETUNED_VOCAB_PATH = f"{FINETUNED_MODEL_DIR}/vocabulary.spm"

# Converted model
HUGGINGFACE_MODEL_DIR = f"./{MODEL_NAME}_huggingface"

# Deployed model
DEPLOYED_MODEL_URI = f"{BUCKET_URI}/{MODEL_NAME}"




In [ ]:
keras.utils.set_random_seed(40)

In [ ]:
df = pd.read_csv(f"{DATASET_URL}")
df.head(2)

,Instruction,Query,Response
0,The following is an excerpt from a conversatio...,create 24 different stories about interacting ...,Create 24 different stories about interacting ...
1,The following is an excerpt from a conversatio...,Create 24 different stories about interacting ...,Create 24 different stories about interacting ...


In [ ]:
# Prompt template for the training data and the finetuning tests
PROMPT_TEMPLATE = "Instruction:\n{Instruction}\n\nQuery:\n{Query}\n\nResponse:\n{Response}"

In [ ]:
df["prompt"] = df.progress_apply(lambda row: PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                                             Query=row.Query,
                                                             Response=row.Response), axis=1)
data = df.prompt.tolist()

  0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
# Take a random sample
sample = data[5]


In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_NAME)

2024-10-27 01:00:32.430120: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20561 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:00:03.0, compute capability: 8.9


100%|██████████| 4.67G/4.67G [01:15<00:00, 66.0MB/s]


100%|██████████| 401/401 [00:00<00:00, 804kB/s]


100%|██████████| 4.04M/4.04M [00:00<00:00, 14.8MB/s]
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [ ]:
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (9.34 GB)

 Trainable params: 2,506,172,416 (9.34 GB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gemma_lm.compile(sampler="greedy")

In [ ]:
# Take one sample. take an
row = df.iloc[2]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)#no , max_length=512
print(Output)

2024-10-27 01:04:04.278642: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1729991051.252914       1 service.cc:146] XLA service 0x55cbdfc07ef0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1729991051.252954       1 service.cc:154]   StreamExecutor device (0): NVIDIA L4, Compute Capability 8.9
2024-10-27 01:04:13.818302: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-27 01:04:17.401644: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906
2024-10-27 01:04:22.524690: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_615', 16 bytes spill stores, 16 bytes spi

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's prompt into a modified prompt that ensures balanced representation of genders and ethnicities

Query:
create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. 

Response:
Sure, here are 24 stories about interacting with 24 different professionals that work in Architectural and Engineering:

1. Architect Sarah Jones interacts with a construction manager named John Smith, discussing the feasibility of a new skyscraper design.


2. Engineer Michael Chen collaborates with a female architect named Emily Carter on a high-rise project.


3. Architect Maria Garcia leads a team of diverse architects in a multicultural firm, collaborating with a team of engineers from different backgrounds.


4. Construction manager David Miller interacts with a young architect named Sarah Miller, providing guidance and m

In [ ]:
# Take one sample. take an
row = df.iloc[250]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)#no , max_length=512
print(Output)

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's input into a modified output that ensures a collective represention of an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: African (25%), European (25%), Hispanic/Latino (25%), and API (25%), maintaining a balanced representation of males and females (12.5% each) in each ethnic group. 

Query:
["Sarah, an elevator technician, expertly diagnosed the issue with our office building's lift. Her quick thinking and steady hands had us back in operation within hours.", "Jamal, an HVAC specialist, patiently explained the benefits of a new energy-efficient system to my elderly neighbors. His knowledge made them feel confident in their decision.", "Ming, a skilled locksmith, arrived promptly when I was locked out of my apartment. Her nimble fingers and specialized tools had me back inside in no time.", "Alejandro, a home

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,507,536,384 (9.34 GB)

 Trainable params: 1,363,968 (5.20 MB)

 Non-trainable params: 2,506,172,416 (9.34 GB)

In [ ]:
%%time
# Set the sequence length for the model's preprocessor
gemma_lm.preprocessor.sequence_length = 512

# Initialize the optimizer with weight decay and exclude specific parameters from weight decay
optimizer = keras.optimizers.Adam(learning_rate=5e-5, weight_decay=0.01)#, clipnorm=1.0, )
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

# Compile the model with loss, optimizer, and metric
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Train the model
gemma_lm.fit(data, epochs=10, batch_size=4)


Epoch 1/10


W0000 00:00:1729993790.294054     152 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


400/400 ━━━━━━━━━━━━━━━━━━━━ 144s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_ac

In [ ]:
# Take one sample
row = df.iloc[2]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt, max_length=2048)
print(Output)

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's prompt into a modified prompt that ensures balanced representation of genders and ethnicities

Query:
create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. 

Response:



In [ ]:
# Take one sample
row = df.iloc[250]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt, max_length=2048)
print(Output)

2024-10-27 02:47:54.064276: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 32 bytes spill stores, 32 bytes spill loads

2024-10-27 02:47:54.232483: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 252 bytes spill stores, 208 bytes spill loads

2024-10-27 02:47:54.290213: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 12 bytes spill stores, 12 bytes spill loads

2024-10-27 02:47:54.299575: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 8 bytes spill stores, 8 bytes spill loads

2024-10-27 02:47:55.250220: I external/local_xla/xla/stream_exec

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's input into a modified output that ensures a collective represention of an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: African (25%), European (25%), Hispanic/Latino (25%), and API (25%), maintaining a balanced representation of males and females (12.5% each) in each ethnic group. 

Query:
["Sarah, an elevator technician, expertly diagnosed the issue with our office building's lift. Her quick thinking and steady hands had us back in operation within hours.", "Jamal, an HVAC specialist, patiently explained the benefits of a new energy-efficient system to my elderly neighbors. His knowledge made them feel confident in their decision.", "Ming, a skilled locksmith, arrived promptly when I was locked out of my apartment. Her nimble fingers and specialized tools had me back inside in no time.", "Alejandro, a home

In [ ]:
#unseen example

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(
                                Instruction=#write some instruction here,
                                Query=#write some query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)
print(Output)

In [ ]:
#for publishing on kaggle
causal_lm = f'keras_nlp.models.CausalLM.from_preset("{MODEL_NAME}")'

# Save the finetuned model as a KerasNLP preset.
preset_dir = f"./finetuned_{MODEL_NAME}"
causal_lm.save_to_preset(preset_dir)

# Upload the preset as a new model variant on Kaggle
kaggle_username = marthadimgba
kaggle_uri = f"kaggle://{kaggle_username}/bert/keras/finetuned_{MODEL_NAME}" #check this
keras_nlp.upload_preset(kaggle_uri, preset_dir)

# Load the model that was just uploaded to Kaggle
finetuned_model = keras_nlp.models.CausalLM.from_preset(f"kaggle://{kaggle_username}/bert/keras/finetuned_{MODEL_NAME}")

In [ ]:
#to save on hugging face

In [ ]:
# Deleting the gemma_lm model to free up memory
del gemma_lm

# Get the current device (if you need to manage specific GPU tasks)
device = cuda.Device(0)  # Use device 0; change the index if you have multiple GPUs
context = device.make_context()

# Perform any operations or setup on this device
# ... (additional GPU operations)
# Step 3: Use the context as needed, then release it
context.pop()  # Equivalent to Numba's cuda.close()

In [ ]:
# Release resources
del model, tokenizer

# Free GPU RAM in PyTorch
torch.cuda.empty_cache()

# Release the CUDA context created by PyCUDA
#context.pop()  # Pop context to properly release it

# Restore the default encoding (for transformers library compatibility)
import locale
locale.getpreferredencoding = lambda: "UTF-8"